# PBS Utah fundraising EDA

Explore the supplied donor data and identify questions and features for later model building. This notebook focuses on giving patterns, channels, donor segments, viewing and cultivation.

Run from the project root. By default, cells load validated saved results; rebuild switches are off. See [README](README.md) for environment setup and full reproduction.

Companion notebooks: [data preparation](data_preparation.ipynb) contains source checks and cleanup; [historical modeling](modeling.ipynb) contains outcome feasibility, model comparison and final evaluation.

## Key findings and scope

- Broad Legacy/Miscellaneous channel labels account for **85.88%** of recorded cash; their meaning needs clarification.
- Current sustainers contribute **86.37%** of latest-window active individual giving. Study dated recurring-payment behavior rather than backfilling current status.
- June has fewer payment records than May in every observed year; verify source coverage before interpreting seasonality.
- Viewing associations change under alternative fiscal-year mappings; engagement is not yet a validated historical predictor.

The exports contain synthetic placeholders. Amounts are source units. Channel totals include organizations and unresolved donors; individual analyses use resolved records. Current labels are descriptive, viewing absence is not zero engagement, and associations do not establish outreach effects. Operational pilot inputs are not required for EDA.

The earlier model test is consumed and is documented separately. This notebook does not train or tune a model.

## 1. Giving trends and donor profiles

Validated Unite payments support recency, frequency, amount, retention and growth profiles. Legacy payments and soft credits are excluded from donor cash features; unresolved Unite payments remain in overall cash totals but not resolved donor profiles.

Periods use explicit dates rather than assumed fiscal-year labels. The default loads a validated run; set `REBUILD_DONOR_EDA = True` only to create a new versioned mart. Implementation: `analysis/donor_eda.py`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from analysis.donor_eda import run_donor_eda, plot_donor_eda

EDA_ROOT = Path.cwd()
if not (EDA_ROOT / 'objectives.md').exists():
    EDA_ROOT = EDA_ROOT.parent
INPUT_CLEANED_RUN = EDA_ROOT / 'data/cleaned/20261010T035947Z_cf744e59'
EDA_RUN = EDA_ROOT / 'data/marts/20261010T043152Z_873c7300'
REBUILD_DONOR_EDA = False

if REBUILD_DONOR_EDA:
    donor_eda = run_donor_eda(EDA_ROOT, INPUT_CLEANED_RUN)
else:
    assert (EDA_RUN / '_SUCCESS').exists()
    donor_eda = {
        'output': EDA_RUN,
        'stats': json.loads((EDA_RUN / 'manifest.json').read_text()),
        'period_summary': pd.read_csv(EDA_RUN / 'period_summary.csv', parse_dates=['period_start','period_end']),
        'thresholds': pd.read_csv(EDA_RUN / 'threshold_sensitivity.csv'),
        'major_class_summary': pd.read_csv(EDA_RUN / 'major_class_summary.csv'),
        'features': pd.read_parquet(EDA_RUN / 'donor_features.parquet'),
        'review': pd.read_parquet(EDA_RUN / 'exploratory_review_cohort.parquet'),
    }
print(json.dumps(donor_eda['stats'], indent=2))
print('EDA outputs:', donor_eda['output'])


### Trends, retention and threshold sensitivity

Retention is the fraction of prior-window positive-giving constituents who gave again in the next window. Counts and giving trends include resolved individuals and organizations; threshold sensitivity and the review cohort include individuals only. 

In [ ]:
summary_view = donor_eda['period_summary'].copy()
summary_view['giving_source_units'] = summary_view.giving_hundredths / 100
summary_view['retention_percent'] = summary_view.retention_fraction * 100
print('Dated 12-month windows:')
print(summary_view[['period_start','period_end','active_constituents','giving_source_units','retained_constituents','retention_percent']].to_string(index=False))
print('\nIllustrative threshold sensitivity, individuals only:')
threshold_view = donor_eda['thresholds'].copy()
threshold_view['giving_source_units'] = threshold_view.giving_hundredths / 100
print(threshold_view.drop(columns='giving_hundredths').to_string(index=False))
print('\nCurrent class labels, restricted to constituents observed in resolved Unite payments:')
print(donor_eda['major_class_summary'].to_string(index=False))
print('\nLatest-window giving patterns, individuals only:')
features_view = donor_eda['features']
print(features_view.loc[features_view.is_individual,'giving_pattern'].value_counts().to_string())
print('\nReview cohort: top 10 rows by latest giving; exploratory, not predicted potential:')
print(donor_eda['review'][['constituent_id','giving_hundredths','previous_giving_hundredths','recency_days','in_latest_recorded_cultivation','major_donor_class_pbs_utah']].head(10).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
donor_figure, donor_plot_path = plot_donor_eda(donor_eda)
plt.show()
print('Chart saved:', donor_plot_path)


### What the donor profiles show

The latest window, **2025-07-01 through 2026-06-30**, contains **4,188,399 source units** of resolved-constituent cash. Retention is **79.2%** (25,999/32,819 prior-window donors); these totals include organizations.

Among **38,163 active individuals**, only **8** gave at least **1,000** source units and **5** at least **2,500**. These illustrative thresholds are not agreed major-donor definitions. The 90th-percentile cutoff of **240** units yields **4,533** individuals because of ties. Of these, **4,151** are absent from FY2025 cultivation selection; that does not establish current assignment status.

The donor profiles exclude **33,733 unresolved payments totaling 433,233 units**, while preserving them in cleaned cash totals. Rare large-gift labels and unresolved attribution should inform later modeling choices.

## 2. Channels and donor segments

Campaign-reference communication method and payment tender are distinct fields. Channel cash summaries include all Unite records; segment and next-window comparisons use resolved individuals. Each historical donor's dominant channel is assigned from baseline cash before measuring later giving. Unique donor counts overlap across channel cash totals and cannot be added.

Default behavior reloads saved aggregates. Implementation: `analysis/extended_eda.py`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from analysis.extended_eda import run_extended_eda, plot_extended_eda

EXTENDED_ROOT = Path.cwd()
if not (EXTENDED_ROOT / 'objectives.md').exists():
    EXTENDED_ROOT = EXTENDED_ROOT.parent
EXTENDED_RUN = EXTENDED_ROOT / 'data/extended_eda/20261011T060942Z_29117690'
REBUILD_EXTENDED_EDA = False
if REBUILD_EXTENDED_EDA:
    extended = run_extended_eda(EXTENDED_ROOT,
        EXTENDED_ROOT / 'data/cleaned/20261010T035947Z_cf744e59',
        EXTENDED_ROOT / 'data/marts/20261010T043152Z_873c7300')
else:
    assert (EXTENDED_RUN / '_SUCCESS').exists()
    extended_manifest = json.loads((EXTENDED_RUN / 'manifest.json').read_text())
    extended = {'output': EXTENDED_RUN, 'manifest': extended_manifest}
    extended.update({name: pd.read_csv(EXTENDED_RUN / (name + '.csv')) for name in extended_manifest['outputs']})
print('Extended EDA:', extended['output'])
print('Payments / cash source units:', extended['manifest']['payment_rows'], extended['manifest']['payment_amount_source_units'])
print('Validation checks passed:', extended['manifest']['validation_checks'])
print('Unmapped channel rows / cash:', extended['manifest']['unmapped_channel_rows'], extended['manifest']['unmapped_amount_source_units'])

### Campaign channels, payment methods and timing

Communication method comes from the campaign reference and differs from payment tender. Neither proves exposure to a solicitation. These tables describe the recorded cash mix; response rates, acquisition efficiency and ROI need denominators and costs not supplied here.

In [ ]:
channels = extended['channel_totals'].copy()
channels['cash_share_percent'] = channels.cash_share * 100
print('Campaign-reference channels, all Unite cash:')
print(channels[['channel','payments','positive_donors','giving_source_units','cash_share_percent','median_payment_source_units']].to_string(index=False))
print('\nPayment tender, all Unite cash:')
print(extended['tender_totals'][['tender_type','payments','positive_donors','giving_source_units','mean_payment_source_units']].to_string(index=False))
campaigns_eda = extended['campaign_totals']
print('\nShare of recorded cash in 10 largest campaign-name/code groups:', round(100 * campaigns_eda.head(10).giving_hundredths.sum() / campaigns_eda.giving_hundredths.sum(),2), '%')
months_eda = extended['calendar_months'].copy()
months_eda['year'] = pd.to_datetime(months_eda.calendar_month).dt.year
months_eda['month'] = pd.to_datetime(months_eda.calendar_month).dt.month
print('\nMay/June payment counts by calendar year:')
print(months_eda.loc[months_eda.month.isin([5,6])].pivot(index='year',columns='month',values='payments').to_string())
print('\nThird-party payment tender: organization/unresolved composition:')
print(extended['tender_constituent_mix'].loc[extended['tender_constituent_mix'].tender_type.eq('Third Party'), ['recorded_constituent_type','payments','positive_donors','giving_source_units']].to_string(index=False))

### Baseline channel and subsequent recorded giving

Each active baseline individual is assigned the channel accounting for the most baseline cash. Equal dominant channels form a separate tie group. Outcomes use the following dated 12-month window; this avoids assigning channels using later payments. Small groups are shown with their denominators and should not be ranked as proven winners.

In [ ]:
transitions = extended['channel_transitions']
latest_transition = transitions.loc[transitions.next_period_end.eq(transitions.next_period_end.max())].copy()
latest_transition['retention_percent'] = latest_transition.retention_fraction * 100
latest_transition['upgrade_percent'] = latest_transition.upgrade_fraction * 100
print('Baseline ending 2025-06-30; outcome window ending 2026-06-30:')
print(latest_transition[['channel','baseline_donors','retained','upgrades','retention_percent','upgrade_percent']].to_string(index=False))
print('\nLatest-window active individual segments; current snapshot labels are descriptive:')
segments_eda = extended['latest_individual_segments']
print(segments_eda[['segment','segment_value','donors','giving_source_units','median_giving_source_units','median_positive_payments']].to_string(index=False))

## 3. Viewing and cultivation

Title totals sum genre-level counts, not confirmed distinct titles. Completion is the median of row-level mean watch fractions, not a pooled completion rate. Missing viewing records remain a separate group.

Calendar sensitivity maps FY labels either to June of the same year or to July-start windows ending the next June. Neither mapping is confirmed. Do not use these joins as historical predictors until calendar semantics are resolved.

Cultivation summaries describe selected cohorts. Selection/capacity ratios do not measure staff hours, and annual contact/holdout flags do not establish randomization.

In [ ]:
print('Viewing by source FY label:')
print(extended['viewing_years'].to_string(index=False))
view_sensitivity = extended['viewing_calendar_sensitivity']
print('\nLatest next-window outcomes under two unconfirmed calendar mappings:')
print(view_sensitivity.loc[view_sensitivity.next_period_end.eq(view_sensitivity.next_period_end.max()), ['calendar_assumption','viewing_group','baseline_donors','retained','upgrades','retention_fraction','upgrade_fraction']].to_string(index=False))
print('\nCultivation selected cohorts by source FY label:')
print(extended['cultivation_portfolio_summary'].groupby('fiscal_year')[['selected','contacted','holdout']].sum().to_string())
print('\nPortfolio selection/capacity ratios:')
print(extended['cultivation_portfolio_summary'][['portfolio_id','selected_to_capacity_ratio','contact_fraction']].groupby('portfolio_id').mean().to_string())

In [ ]:
import matplotlib.pyplot as plt
extended_figure, extended_chart_path = plot_extended_eda(extended)
plt.show()
print('Descriptive chart:', extended_chart_path)

## 4. Interpretation and next EDA questions

| Observation | Implication |
| --- | --- |
| Legacy 59.23%, Miscellaneous 26.65%, Digital 10.25% of all cash | Clarify broad communication labels before treating them as actionable channels |
| Third Party tender: 733 payments, 719 organization records and 14 unresolved | Its 466.42-unit mean payment does not characterize individual prospects |
| June 35,662 payment rows versus May 148,061 across six years | Investigate source date-generation or coverage before adding seasonal effects |
| Current sustainers: median 74 units and 12 positive payments; others 10 units and 1 payment | Examine dated recurring-payment features; current labels are not historical facts |
| Digital baseline retention 3,330/4,149 (80.26%); Legacy 12,490/15,834 (78.88%) | Compare donor composition; these are descriptive differences, not channel effects |
| Phone has only 3 upgrades among 28 baseline donors | Keep denominators visible; small groups are unstable |
| Recorded viewers rise 25.46% from FY2025 to FY2026 | Check population and coverage changes before interpreting engagement growth |
| Each cultivation year has 1,000 selected, 752 contacted and 248 holdouts | Highly structured synthetic patterns do not establish workloads or intervention effects |

**Next source checks:** channel-category meanings, payment-date coverage and FY-label semantics. Then explore organization-funded cash separately and use dated payment history for recurring-giving comparisons. Existing data supports these tasks without current contact permissions or staffing budgets.

All **42 validation checks** passed, and **13 aggregate tables** reproduced exactly. See [the findings report](analysis/extended_eda_report.md) for detail and [README](README.md) for reproduction. No new model has been fitted or independently validated by this post-test EDA.